# Self-Representation Experiment — Full Pipeline
Run this notebook on **Google Colab** (Runtime → Change runtime type → T4 GPU).

Steps:
1. Setup environment
2. Clone repo & install dependencies
3. Authenticate with HuggingFace
4. Mount Google Drive (to persist outputs)
5. Run the full pipeline


## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — make sure you selected a GPU runtime.')

## 1. Mount Google Drive (persists outputs across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ureca26_outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Outputs will be saved to: {DRIVE_DIR}')

## 2. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/ureca26.git'  # <-- update this
REPO_DIR = '/content/ureca26'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install dependencies (takes ~3 min on first run)
!pip install -e . -q

## 3. HuggingFace login
You need a HF token with access to `meta-llama/Llama-3.1-8B-Instruct`.
Get one at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Option A: store token as a Colab secret named HF_TOKEN (Secrets tab in left sidebar)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in via Colab secret.')
except Exception:
    # Option B: paste token directly
    login()  # will prompt interactively

## 4. Configure output paths to Google Drive

In [ ]:
import yaml, os

CONFIG_PATH = 'configs/experiment.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Point all output paths to Google Drive so they survive session restarts
cfg['paths']['data_dir']        = f'{DRIVE_DIR}/data'
cfg['paths']['activations_dir'] = f'{DRIVE_DIR}/data/activations'
cfg['paths']['directions_dir']  = f'{DRIVE_DIR}/data/directions'
cfg['paths']['figures_dir']     = f'{DRIVE_DIR}/figures'
cfg['paths']['prompts_file']    = f'{DRIVE_DIR}/data/prompts.json'

COLAB_CONFIG = '/tmp/experiment_colab.yaml'
with open(COLAB_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

# Create output dirs
for d in cfg['paths'].values():
    if not d.endswith('.json'):
        os.makedirs(d, exist_ok=True)

print('Config written to', COLAB_CONFIG)
print('Output dirs created under', DRIVE_DIR)

## 5. Step 01 — Generate prompts

In [ ]:
!python scripts/01_generate_prompts.py --config {COLAB_CONFIG}

## 6. Step 02 — Extract activations
This is the longest step (~15–20 min on T4). The model (~16GB float16) fits on T4 (16GB VRAM) but is tight. If you get OOM, uncomment the `--batch_size 4` line.

In [ ]:
!python scripts/02_extract_activations.py \
    --config {COLAB_CONFIG} \
    --batch_size 4
    # increase to 8 if you have an A100/A10G

## 7. Step 03 — Find directions & probes

In [ ]:
!python scripts/03_find_directions.py --config {COLAB_CONFIG}

## 8. Step 04 — Test specificity

In [ ]:
!python scripts/04_test_specificity.py --config {COLAB_CONFIG}

## 9. Step 05 — Visualize

In [ ]:
!python scripts/05_visualize.py --config {COLAB_CONFIG}

# Display figures inline
import glob
from IPython.display import Image, display

figs = sorted(glob.glob(f"{DRIVE_DIR}/figures/**/*.png", recursive=True))
print(f'Found {len(figs)} figures:')
for fig in figs:
    print(' ', fig)
    display(Image(fig))

## Done!
All outputs are saved to your Google Drive under `ureca26_outputs/`.
They will persist even after this Colab session ends.